In [37]:
import polars as pl
import datetime as dt
from pathlib import Path

In [38]:
pl.Config.set_tbl_cols(200)               
pl.Config.set_tbl_rows(50)                

polars.config.Config

In [39]:
def save_clean_df(df, name, base_dir = "data/clean", country="US", data_type="trade", fmt = "parquet"):
    """
    Save a cleaned DataFrame in data/clean/.
    Create the directory if it does not exist.
    """

    # Name handling
    if name is None or name == "":
        name = "cleaned_data"

    # Create directory if it does not exist
    if country == "merged":
        save_dir = Path(base_dir) / country 
    else:
        save_dir = Path(base_dir) / country / data_type
    save_dir.mkdir(parents=True, exist_ok=True)

    # Handle existing files
    file_path = save_dir / f"{name}.{fmt}"
    i = 1
    while file_path.exists():
        file_path = save_dir / f"{name}_{i}.{fmt}"
        i += 1

    # Save DataFrame
    if fmt == "parquet":
        df.write_parquet(file_path)
    elif fmt == "csv":
        df.write_csv(file_path)
    else:
        raise ValueError("Format must be 'parquet' or 'csv'")

    print(f"Saved {name} to {file_path}")
    return

In [40]:
def import_df_trade(path, label = None, save = True, save_name = None, base_dir = "data/clean", file_format="parquet", country="US"):
    """
    Import and clean trade data from a given path.
    """

    origin = dt.datetime(1899, 12, 30)
    label = "" if label is None else f"_{label}"

    # Load data
    if file_format == "parquet":
        df = pl.scan_parquet(path).collect()
    elif file_format == "csv":
        df = pl.scan_csv(path).collect()
    else:
        raise ValueError("file_format must be 'parquet' or 'csv'")

    # Clean & rename
    df = df.rename({df.columns[0]: "time"})
    df = df.drop(["trade-stringflag", "trade-rawflag"]).drop_nulls(subset=["time", "trade-price", "trade-volume"])
    df = df.rename({"trade-price": f"price{label}", "trade-volume": f"volume{label}"})

    # Datetime conversion
    df = df.with_columns(pl.col("time").map_elements(lambda x: origin + dt.timedelta(days=float(x))).alias("datetime")).drop("time")
    if country == "US":
        df = df.with_columns(pl.col("datetime").dt.replace_time_zone("UTC").dt.convert_time_zone("America/New_York"))
        df = df.filter(pl.col("datetime").dt.time().is_between(pl.time(9, 30), pl.time(16, 0))).sort("datetime").select(["datetime", f"price{label}", f"volume{label}"])
    elif country == "FR":
        df = df.with_columns(pl.col("datetime").dt.replace_time_zone("UTC").dt.convert_time_zone("Europe/Paris"))
        df = df.filter(pl.col("datetime").dt.time().is_between(pl.time(9, 0), pl.time(17, 30))).sort("datetime").select(["datetime", f"price{label}", f"volume{label}"])
    else:
        raise ValueError("country must be 'US' or 'FR'")
    
    # Aggregate duplicates
    df = df.group_by("datetime").agg([pl.col(f"price{label}").mean().alias(f"price{label}"),pl.col(f"volume{label}").mean().alias(f"volume{label}"),]).sort("datetime")
    
    # Compute log-returns
    df = df.with_columns((pl.col(f"price{label}").log() - pl.col(f"price{label}").log().shift(1)).alias(f"return{label}"))

    # Save cleaned DataFrame
    if save:
        if save_name is None:
            save_name = f"trade{label}"
        save_clean_df(df=df, name=save_name, base_dir=base_dir, country=country, data_type="trade", fmt=file_format)

    return df


In [41]:
def import_df_BBO(path, label = None, save = True, save_name = None, base_dir = "data/clean", file_format="parquet", country="US"):
    """
    Import and clean trade data from a given path.
    """

    origin = dt.datetime(1899, 12, 30)
    label = "" if label is None else f"_{label}"

    # Load data
    if file_format == "parquet":
        df = pl.scan_parquet(path).collect()
    elif file_format == "csv":
        df = pl.scan_csv(path).collect()
    else:
        raise ValueError("file_format must be 'parquet' or 'csv'")
    
    # Clean & rename
    df = df.rename({df.columns[0]: "time"})
    df = df.drop_nulls(subset=["bid-price", "bid-volume", "ask-price", "ask-volume"])
    df = df.rename({"bid-price": f"bid_price{label}", "bid-volume": f"bid_volume{label}", "ask-price": f"ask_price{label}", "ask-volume": f"ask_volume{label}"})

    # Datetime conversion
    df = df.with_columns(pl.col("time").map_elements(lambda x: origin + dt.timedelta(days=float(x))).alias("datetime")).drop("time")
    if country == "US":
        df = df.with_columns(pl.col("datetime").dt.replace_time_zone("UTC").dt.convert_time_zone("America/New_York"))
        df = df.filter(pl.col("datetime").dt.time().is_between(pl.time(9, 30), pl.time(16, 0))).sort("datetime").select(["datetime", f"bid_price{label}", f"bid_volume{label}", f"ask_price{label}", f"ask_volume{label}"])
    elif country == "FR":
        df = df.with_columns(pl.col("datetime").dt.replace_time_zone("UTC").dt.convert_time_zone("Europe/Paris"))
        df = df.filter(pl.col("datetime").dt.time().is_between(pl.time(9, 0), pl.time(17, 30))).sort("datetime").select(["datetime", f"bid_price{label}", f"bid_volume{label}", f"ask_price{label}", f"ask_volume{label}"])
    else:
        raise ValueError("country must be 'US' or 'FR'")
    
    # Aggregate duplicates
    df = df.group_by("datetime").agg([pl.col(f"bid_price{label}").mean().alias(f"bid_price{label}"),pl.col(f"bid_volume{label}").mean().alias(f"bid_volume{label}"),pl.col(f"ask_price{label}").mean().alias(f"ask_price{label}"),pl.col(f"ask_volume{label}").mean().alias(f"ask_volume{label}")]).sort("datetime")
    
    # Compute spread
    df = df.with_columns((pl.col(f"ask_price{label}") - pl.col(f"bid_price{label}")).alias(f"spread{label}"))

    # Save cleaned DataFrame
    if save:
        if save_name is None:
            save_name = f"bbo{label}"
        save_clean_df(df=df, name=save_name, base_dir=base_dir, country=country, data_type="bbo", fmt=file_format)

    return df


In [42]:
def bucket_time_trade(df, freq, price_col = "price", volume_col = "volume", return_col = "return", time_col = "datetime") :
    """
    Aggregate trades into time buckets.
    """
    
    return (
        df
        .group_by_dynamic(
            time_col,
            every=freq,
            period=freq,
            closed="left",
            label="left",
        )
        .agg([
            pl.col(price_col).mean().alias(price_col),
            pl.col(volume_col).sum().alias(volume_col),
            pl.col(return_col).sum().alias(return_col), # Sum of log-returns 
        ])
        .sort(time_col)
    )


In [43]:
def bucket_time_BBO(df, freq, bid_price_col = "bid-price", bid_volume_col = "bid-volume", ask_price_col = "ask-price", ask_volume_col = "ask-volume", spread_col = "spread", time_col = "datetime") :
    """
    Aggregate trades into time buckets.
    """
    
    return (
        df
        .group_by_dynamic(
            time_col,
            every=freq,
            period=freq,
            closed="left",
            label="left",
        )
        .agg([
            pl.col(bid_price_col).mean().alias(bid_price_col),
            pl.col(bid_volume_col).sum().alias(bid_volume_col),
            pl.col(ask_price_col).mean().alias(ask_price_col),
            pl.col(ask_volume_col).sum().alias(ask_volume_col),
            pl.col(spread_col).mean().alias(spread_col),
        ])
        .sort(time_col)
    )


In [44]:
def merging_df(df, assets, save = False, save_name = "merged_data", base_dir = "data/clean"):
    """
    Merge multiple DataFrames on datetime.
    """

    for a in assets:
        df[a] = df[a].with_columns(
            pl.col("datetime").dt.convert_time_zone("UTC")
        )

    df_merged = df[assets[0]]
    for asset in assets[1:]:
        df_merged = df_merged.join(df[asset], on="datetime", how="inner")

    if save:
        if save_name is None:
            save_name = "merged_data"
        save_clean_df(df=df_merged, name=save_name, base_dir=base_dir, country="merged", data_type="merged", fmt="parquet")

    return df_merged

In [45]:
def normalzing_df(df):
    """
    Normalize the return columns of a DataFrame.
    """

    for col in df.columns:
        if df[col].dtype == pl.Float64 :
            mean = df[col].mean()
            std = df[col].std()
            df = df.with_columns(((pl.col(col) - mean) / std).alias(f"{col}_normalized"))
    
    return df

In [46]:
base_dir = "../data/clean"
assets = ["AAPL", "SPY"]
assets_BBO = ["AAPL", "AMZN", "MSFT", "GOOGL"]
df_trade = {}
df_BBO = {}
df_trade_min = {}
df_trade_sec = {}
df_BBO_min = {}
df_BBO_sec = {}
dict_asset_exchange = {"AAPL": "OQ", "SPY": "P", "AMZN": "OQ", "MSFT": "OQ", "GOOGL": "OQ"}
dict_country_asset = {"AAPL": "US", "SPY": "US", "AMZN": "US", "MSFT": "US", "GOOGL": "US"}
dict_format_asset = {"AAPL": "parquet", "SPY": "parquet", "AMZN": "parquet", "MSFT": "parquet", "GOOGL": "parquet"}
save_asset = True
save_merge = True
data_type_vec = ["trade", "BBO"]


In [ ]:
for data_type in data_type_vec :
    for asset in assets :
        save_name = f"{data_type}_{asset}"
        exchange = dict_asset_exchange[asset]
        country = dict_country_asset[asset]
        file_format = dict_format_asset[asset]
        path = f"../data/raw/{country}/{data_type}/{asset}.{exchange}/*.{file_format}"

        if data_type == "trade" :
            price_col = f"price_{asset}"
            return_col = f"return_{asset}"
            volume_col = f"volume_{asset}"

            df_trade[asset] = import_df_trade(path=path, label=asset, save=save_asset, save_name=save_name, base_dir=base_dir, file_format=file_format, country=country)

            df_trade_min[asset] = normalzing_df(bucket_time_trade(df=df_trade[asset], freq="1m", price_col=price_col, volume_col=volume_col, return_col=return_col, time_col="datetime"))
            df_trade_sec[asset] = normalzing_df(bucket_time_trade(df=df_trade[asset], freq="1s", price_col=price_col, volume_col=volume_col, return_col=return_col, time_col="datetime"))

        else :
            bid_price_col = f"bid_price_{asset}"
            bid_volume_col = f"bid_volume_{asset}"
            ask_price_col = f"ask_price_{asset}"
            ask_volume_col = f"ask_volume_{asset}"
            spread_col = f"spread_{asset}"

            df_BBO[asset] = import_df_BBO(path=path, label=asset, save=save_asset, save_name=save_name, base_dir=base_dir, file_format=file_format, country=country)
            df_BBO_min[asset] = normalzing_df(bucket_time_BBO(df=df_BBO[asset], freq="1m", bid_price_col=bid_price_col, bid_volume_col=bid_volume_col, ask_price_col=ask_price_col, ask_volume_col=ask_volume_col, spread_col=spread_col, time_col="datetime"))
            df_BBO_sec[asset] = normalzing_df(bucket_time_BBO(df=df_BBO[asset], freq="1s", bid_price_col=bid_price_col, bid_volume_col=bid_volume_col, ask_price_col=ask_price_col, ask_volume_col=ask_volume_col, spread_col=spread_col, time_col="datetime"))

df_trade_min_merged = merging_df(df=df_trade_min, assets=assets, save=save_merge, save_name="merged_1min", base_dir=base_dir)
df_trade_sec_merged = merging_df(df=df_trade_sec, assets=assets, save=save_merge, save_name="merged_1sec", base_dir=base_dir)
df_BBO_min_merged = merging_df(df=df_BBO_min, assets=assets, save=save_merge, save_name="merged_BBO_1min", base_dir=base_dir)
df_BBO_sec_merged = merging_df(df=df_BBO_sec, assets=assets, save=save_merge, save_name="merged_BBO_1sec", base_dir=base_dir)

In [ ]:
data_type = data_type_vec[1]
for asset in assets_BBO :
    save_name = f"{data_type}_{asset}"
    exchange = dict_asset_exchange[asset]
    country = dict_country_asset[asset]
    file_format = dict_format_asset[asset]
    path = f"../data/raw/{country}/{data_type}/{asset}.{exchange}/*.{file_format}"

    bid_price_col = f"bid_price_{asset}"
    bid_volume_col = f"bid_volume_{asset}"
    ask_price_col = f"ask_price_{asset}"
    ask_volume_col = f"ask_volume_{asset}"
    spread_col = f"spread_{asset}"

    df_BBO[asset] = import_df_BBO(path=path, label=asset, save=save_asset, save_name=save_name, base_dir=base_dir, file_format=file_format, country=country)
    df_BBO_min[asset] = normalzing_df(bucket_time_BBO(df=df_BBO[asset], freq="1m", bid_price_col=bid_price_col, bid_volume_col=bid_volume_col, ask_price_col=ask_price_col, ask_volume_col=ask_volume_col, spread_col=spread_col, time_col="datetime"))
    df_BBO_sec[asset] = normalzing_df(bucket_time_BBO(df=df_BBO[asset], freq="1s", bid_price_col=bid_price_col, bid_volume_col=bid_volume_col, ask_price_col=ask_price_col, ask_volume_col=ask_volume_col, spread_col=spread_col, time_col="datetime"))

df_BBO_min_merged_4assets = merging_df(df=df_BBO_min, assets=assets_BBO, save=save_merge, save_name="merged_BBO_1min_4assets", base_dir=base_dir)
df_BBO_sec_merged_4assets = merging_df(df=df_BBO_sec, assets=assets_BBO, save=save_merge, save_name="merged_BBO_1sec_4assets", base_dir=base_dir)

Saved BBO_AAPL to ..\data\clean\US\bbo\BBO_AAPL_3.parquet
Saved BBO_AMZN to ..\data\clean\US\bbo\BBO_AMZN_1.parquet
Saved BBO_MSFT to ..\data\clean\US\bbo\BBO_MSFT_1.parquet
Saved BBO_GOOGL to ..\data\clean\US\bbo\BBO_GOOGL_1.parquet
Saved merged_BBO_1min_4assets to ..\data\clean\merged\merged_BBO_1min_4assets_1.parquet
Saved merged_BBO_1sec_4assets to ..\data\clean\merged\merged_BBO_1sec_4assets_1.parquet


In [ ]:
df_min_BBO_AAPL = pl.read_parquet("../data/clean/US/bbo/BBO_AAPL_3.parquet")
df_min_BBO_AMZN = pl.read_parquet("../data/clean/US/bbo/BBO_AMZN.parquet")
print(df_min_BBO_AAPL.select("datetime").head())
print(df_min_BBO_AAPL.select("datetime").tail())
print(df_min_BBO_AMZN.select("datetime").head())
print(df_min_BBO_AMZN.select("datetime").tail())

shape: (5, 1)
┌────────────────────────────────┐
│ datetime                       │
│ ---                            │
│ datetime[μs, America/New_York] │
╞════════════════════════════════╡
│ 2009-01-02 09:30:00.128 EST    │
│ 2009-01-02 09:30:00.322 EST    │
│ 2009-01-02 09:30:00.332 EST    │
│ 2009-01-02 09:30:00.633 EST    │
│ 2009-01-02 09:30:00.698 EST    │
└────────────────────────────────┘
shape: (5, 1)
┌────────────────────────────────┐
│ datetime                       │
│ ---                            │
│ datetime[μs, America/New_York] │
╞════════════════════════════════╡
│ 2009-12-31 15:59:59.780 EST    │
│ 2009-12-31 15:59:59.959999 EST │
│ 2009-12-31 15:59:59.970999 EST │
│ 2009-12-31 15:59:59.980 EST    │
│ 2009-12-31 15:59:59.981 EST    │
└────────────────────────────────┘
shape: (5, 1)
┌────────────────────────────────┐
│ datetime                       │
│ ---                            │
│ datetime[μs, America/New_York] │
╞════════════════════════════════╡
│ 2016-01-04 

In [ ]:
df_BBO_min_merged_4assets.head()

datetime,bid_price_AAPL,bid_volume_AAPL,ask_price_AAPL,ask_volume_AAPL,spread_AAPL,bid_price_AAPL_normalized,bid_volume_AAPL_normalized,ask_price_AAPL_normalized,ask_volume_AAPL_normalized,spread_AAPL_normalized,bid_price_AMZN,bid_volume_AMZN,ask_price_AMZN,ask_volume_AMZN,spread_AMZN,bid_price_AMZN_normalized,bid_volume_AMZN_normalized,ask_price_AMZN_normalized,ask_volume_AMZN_normalized,spread_AMZN_normalized,bid_price_MSFT,bid_volume_MSFT,ask_price_MSFT,ask_volume_MSFT,spread_MSFT,bid_price_MSFT_normalized,bid_volume_MSFT_normalized,ask_price_MSFT_normalized,ask_volume_MSFT_normalized,spread_MSFT_normalized,bid_price_GOOGL,bid_volume_GOOGL,ask_price_GOOGL,ask_volume_GOOGL,spread_GOOGL,bid_price_GOOGL_normalized,bid_volume_GOOGL_normalized,ask_price_GOOGL_normalized,ask_volume_GOOGL_normalized,spread_GOOGL_normalized
"datetime[μs, UTC]",f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
